# SSSUMO — Training

Train the submovement detector from scratch, or fine-tune it on the statistics of
real human motion data.

Training runs in two stages:

1. **Pretraining** on purely synthetic data, where every submovement onset,
   amplitude and duration is known. `configs/config-0423-ModGaussian_ampl.yaml`.
2. **Semi-supervised fine-tuning**, where the model is run over real recordings,
   the submovements it detects are pooled into per-dataset statistics, and a
   labelled generator is re-parameterised from those statistics. Real signals are
   never used as targets — only the shape of their statistics is.
   `configs/config-0425-tune_ModGauss_wo_writing.yaml`.

The loop itself lives in `sssumo.training.train`; this notebook sets up the
environment and calls it. For long unattended runs driven from a terminal, see
[`scripts/colab/README.md`](https://github.com/dolphin-in-a-coma/sssumo/blob/main/scripts/colab/README.md).

**Before you start:** enable a GPU with *Runtime → Change runtime type → T4 GPU*.

## Setup

Clone the repository and install the package.

In [ ]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %cd /content/
    if not os.path.exists("sssumo"):
        !git clone https://github.com/dolphin-in-a-coma/sssumo.git
    %cd sssumo
    !pip install -q .
    root_dir = "/content/sssumo"
else:
    root_dir = ".."  # NOTE: assuming you're running this from ./notebooks/

print("root_dir:", root_dir)

### Download the data

The seven tangential-velocity datasets (~1.9 GB) come from the same public
archive the inference notebook uses. Skip this if you only want to train on
synthetic data with the organic evaluation disabled.

In [ ]:
datasets_dir = os.path.join(root_dir, "data")

if IN_COLAB and not os.path.exists(f"{datasets_dir}/steering_tangential_velocity_data.csv"):
    !mkdir -p data
    !curl -L "https://datacloud.helsinki.fi/public.php/dav/files/bE7SP7xCrSyApeC/?accept=zip" -o data/sssumo_data.zip
    !unzip -oq data/sssumo_data.zip -d data/
    !mv data/sssumo_data/* data/
    !rm -r data/sssumo_data data/sssumo_data.zip

print(sorted(f for f in os.listdir(datasets_dir) if f.endswith(".csv")))

### Check the GPU

Training on CPU is impractically slow — a single epoch takes minutes on a T4 and
far longer without one.

In [ ]:
import torch

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU")

## Configure the run

`QUICK_DEMO` keeps the notebook to a few minutes so it completes end to end.
A full pretraining run is **25 epochs of 1000 steps, about 4.3 hours on a T4** —
longer than a free Colab session usually survives. For that, either mount Drive
below so checkpoints persist and use `resume=True` after a disconnect, or drive
the run from a terminal with `scripts/colab/`.

`experiment_name` determines the checkpoint filenames
(`weights/<experiment_name>_<epoch>.pth`). Keep it distinct from any released
checkpoint, or the final epoch will overwrite it.

In [ ]:
from sssumo.utils import Config

QUICK_DEMO = True          # False for a full run
FINE_TUNE = False          # True to fine-tune from the pretrained checkpoint
USE_WANDB = False          # needs a WANDB_KEY Colab secret, or a local login

if FINE_TUNE:
    config_name = "config-0425-tune_ModGauss_wo_writing.yaml"
else:
    config_name = "config-0423-ModGaussian_ampl.yaml"

config = Config(os.path.join(root_dir, "configs", config_name), root_dir=root_dir)
config.experiment_name = ("demo-" if QUICK_DEMO else "") + config_name.replace(".yaml", "")

if QUICK_DEMO:
    config.num_samples = 20    # optimizer steps per epoch (1000 in a full run)
    config.num_epochs = 2      # 25 pretraining / 10 fine-tuning in a full run

print(f"{config.experiment_name}: {config.num_epochs} epochs x {config.num_samples} steps "
      f"x {config.batch_size} trials on {config.device}")

Fine-tuning starts from the pretrained checkpoint, which the config looks for in
`weights/`. The repository ships it in `checkpoints/`, so copy it across.

In [ ]:
import shutil

os.makedirs(os.path.join(root_dir, "weights"), exist_ok=True)
os.makedirs(os.path.join(root_dir, "logs"), exist_ok=True)

if FINE_TUNE:
    src = os.path.join(root_dir, "checkpoints", "config-0423-ModGaussian_ampl_24.pth")
    dst = os.path.join(root_dir, "weights", "config-0423-ModGaussian_ampl_24.pth")
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copyfile(src, dst)
        print("copied pretrained weights to", dst)

### Optional: keep checkpoints after the session ends

`/content` is deleted when the Colab VM is reclaimed. Mounting Drive and pointing
`root_dir` at it makes each epoch's checkpoint durable — worth it for any run
longer than a few minutes. Skip this cell to keep everything ephemeral.

In [ ]:
PERSIST_TO_DRIVE = False

if IN_COLAB and PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    persist_dir = "/content/drive/MyDrive/sssumo_runs"
    for sub in ("weights", "logs"):
        os.makedirs(f"{persist_dir}/{sub}", exist_ok=True)
        local = os.path.join(root_dir, sub)
        if not os.path.islink(local):
            shutil.rmtree(local, ignore_errors=True)
            os.symlink(f"{persist_dir}/{sub}", local)
    print("checkpoints and logs now write to", persist_dir)

### Optional: experiment tracking

With `USE_WANDB = True`, losses, onset metrics and reconstruction plots go to
[Weights & Biases](https://wandb.ai). Store your key as a Colab secret named
`WANDB_KEY` (the key icon in the sidebar). Without it, the same numbers are still
written to `logs/<experiment_name>.txt`.

In [ ]:
import wandb

if USE_WANDB:
    wandb_key = None
    if IN_COLAB:
        from google.colab import userdata
        try:
            wandb_key = userdata.get("WANDB_KEY")
        except Exception:
            print("No WANDB_KEY secret found; falling back to $WANDB_KEY")
    wandb_key = wandb_key or os.getenv("WANDB_KEY")

    wandb.login(key=wandb_key)
    del wandb_key
    wandb.init(project="submovement_detector", name=config.experiment_name,
               config=config.to_dict())

## Train

`train` writes a checkpoint after every epoch and prints per-iteration losses
along with onset precision, recall and mean onset distance.

Two arguments worth knowing:

- `organic_eval_every` — epochs between evaluations on the real datasets. Each
  one loads all seven CSVs at three noise levels, so it takes a few minutes.
  Set it to `0` to train on synthetic data alone.
- `eval_datapoints` — trials per evaluation. `None` uses every trial, as the
  published run did. Values below about 32 can leave a pooled-statistics column
  entirely NaN, which raises `ValueError: 'a' cannot be empty`.

In [ ]:
from sssumo.training import train

checkpoint = train(
    config,
    organic_eval_every=1 if QUICK_DEMO else 5,
    eval_datapoints=64 if QUICK_DEMO else None,
    resume=False,          # True to continue from the newest checkpoint in weights/
    plot=True,             # draw reconstructions inline
)

print("last checkpoint:", checkpoint)

In [ ]:
if USE_WANDB:
    wandb.finish()

## Next steps

- **Evaluate** the result with `notebooks/Analysis - organic and synth.ipynb`, or
  run inference with `notebooks/Inference.ipynb` pointed at your checkpoint.
- **Ablations** — `configs/` holds the variants from the article: shorter kernels,
  no noise, no reconstruction loss, absolute velocity, and leave-one-dataset-out
  fine-tuning runs. Each is a drop-in replacement for `config_name` above.
- **Long runs** — drive them from a terminal with `scripts/colab/`, which keeps a
  session alive, streams the log, and resumes after a lost VM.

### A note on licensing

Training on a mixture that includes `tablet_writing` produces weights covered by
that dataset's research-only licence, so they cannot be released under CC BY 4.0.
That is why `config-0425-tune_ModGauss_wo_writing.yaml` is the released
fine-tuning config. Pretraining is unaffected: it uses only synthetic data, and
`tablet_writing` is read during evaluation only, where no gradient touches it.